In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool("web_search", description = "Search on the Internet for up-to-date information")
def web_search(query: str) -> Dict[str, Any]:
    return tavily_client.search(query)

In [6]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model = "google_genai:gemini-2.5-flash", # Choose the model 
    tools = [web_search],       # Give more power, and accessibility for the agent
    system_prompt = system_prompt, # Maintain the chatbot functionalities
    checkpointer = InMemorySaver() # Retain conversation context
)

In [9]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [10]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [11]:
from langchain.messages import HumanMessage

input = HumanMessage(content=[
    {"type": "text", "text": "I have some leftover chicken and rice. I also have some ingredients in the fridge as shown in the image. What can I make?"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [input]},
    config = {"configurable": {"thread_id": "1"}} 
)

print(response['messages'][-1].content)

[{'type': 'text', 'text': "Based on your leftover chicken and rice, and the fresh ingredients in your fridge, here are a few recipe suggestions:\n\n1.  **Chicken and Vegetable Stir-fry with Rice:** This is a quick and versatile option. You can use your leftover chicken and rice, and incorporate bell peppers, broccoli, carrots, and red onion. You could even add some zucchini and a squeeze of lemon or lime for freshness.\n2.  **Chicken and Rice Skillet with Mixed Vegetables:** Similar to a stir-fry, this one-pan meal can feature your chicken, rice, bell peppers, zucchini, broccoli, carrots, and red onion.\n3.  **Creamy Chicken and Broccoli Rice Casserole:** If you're looking for a comforting dish, you can combine your chicken, rice, and broccoli with some of the milk and cheese from your fridge to create a delicious casserole. Carrots or bell peppers could also be added.\n4.  **Chicken Fried Rice:** A classic way to use leftover rice and chicken! You can add eggs, carrots, bell peppers, 

In [12]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'I have some leftover chicken and rice. I also have some ingredients in the fridge as shown in the image. What can I make?'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/png'}], additional_kwargs={}, response_metadata={}, id='32075b9b-8f85-46e5-9a97-893eae5e6304'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'web_search', 'arguments': '{"query": "chicken and rice recipes with broccoli carrots bell peppers onion"}'}, '__gemini_function_call_thought_signatures__': {'f15b5318-b588-42d3-9a06-5e418ae95d73': 'CtMNAQw51scJVtW+i/AUkwpv0cmR6iSsv8hadbiLbGC5JSmy4x1dfLDtqnmnGz/qencSv+Dus2yfaIT6hxx86aWR2EQ7ldHkp44amdd646i63/IkviTr/lUIwBSsMWJZACVSQDJXpMu0OFNvQbl67G52Knxs2xzrRK36jjZPo2l3R1mepAFcD7mqqG3eEvJpr2KYXgNV2FMxopSnbci/VbBlJMOrBhi3soq4741R84FzVLKqmG5Ngz+riwxbzhCHOxMbwECkEBf/hKRWJAMHysRSwxvYYhLqshbjPQFlbrIsvwpM72YjFsrpfEdnCRTe4XHMpd4dv5Q62VFzDfmoRLwlepO+Rpw97L1BFh4NEvP2FkIzgXSR0syMaTVejm